In [ ]:
## Script for processing of PsychoPy data and output files ready for use with GLMSingle
# Run with python 3.10 fMRI env
# James Fodor 2023

import numpy as np
import csv
import pandas as pd
import json

# set display options
np.set_printoptions(precision=5, threshold=5000, linewidth=220, suppress=True, floatmode='fixed')
pd.set_option('display.max_rows', 500)

### Load results from psychopy output file

In [ ]:
# Load psychopy log file into numpy array
subject_id = '37'
rows = []
with open('../Data-behavioural/Psychopy/sub-'+subject_id+'/all_runs.csv', 'r') as file:
    for row in csv.reader(file):
        rows.append([cell.replace('\n', ' ') for cell in row])
psychopy_data_np = np.array(rows)
print(psychopy_data_np.shape)

# Define params
num_runs = 8
TR = 0.8
num_TR = 636

(435, 72)


In [26]:
# Save as pandas dataframe
start_index = 0 # in case we need to exclude some columns
df = pd.DataFrame(psychopy_data_np[2:,start_index:], columns=psychopy_data_np[0,start_index:])

# Get number of sentences
sentence_ids = df['sentence_id'].values
sentence_ids = sentence_ids[sentence_ids.astype(bool)]
sentence_ids = sentence_ids.astype(int)
num_sentences = np.max(sentence_ids)

# Show all available data
print(df.columns)

Index(['﻿run_file', 'intro_text', 'final_fixation_time', 'sentence_id',
       'sentence_text', 'sentence_duration', 'interstim_duration',
       'question_duration', 'question_text', 'answer_index', 'option_1',
       'option_2', 'option_3', 'option_4', 'runs.thisRepN', 'runs.thisTrialN',
       'runs.thisN', 'runs.thisIndex', 'trials.thisRepN', 'trials.thisTrialN',
       'trials.thisN', 'trials.thisIndex', 'welcome_text.started',
       'mouse.started', 'mouse.x', 'mouse.y', 'mouse.leftButton',
       'mouse.midButton', 'mouse.rightButton', 'mouse.time',
       'ready_key_resp.started', 'next_run_text.started',
       'ready_key_resp.keys', 'key_resp.started',
       'fixation_text_trigger.started', 'key_resp.keys', 'key_resp.rt',
       'key_resp.duration', 'sentence.started', 'sentence.stopped',
       'question_fulltext.started', 'question_fulltext.stopped',
       'question_option_1.started', 'question_option_1.stopped',
       'question_option_2.started', 'question_option_2.sto

### Print results in array for visualisation

In [27]:
# Display key rows
desired_rows = ['runs.thisIndex','key_resp.keys','fixation_text_trigger.started','key_resp.rt',
                'sentence_id','sentence.started','sentence.stopped',
                'question_fulltext.started','question_response.rt','question_fulltext.stopped','answer_index','question_response.keys']
                # 'fixation_text.started','fixation_text.stopped']
                # 'end_run_fixation.started','end_run_fixation.stopped','participant']
df[desired_rows]

,runs.thisIndex,key_resp.keys,fixation_text_trigger.started,key_resp.rt,sentence_id,sentence.started,sentence.stopped,question_fulltext.started,question_response.rt,question_fulltext.stopped,answer_index,question_response.keys
0,0,t,45.11210729996674,30.81798330001766,34,75.98053329996765,82.42537650000304,82.42537650000304,,82.42537650000304,None,None
1,0,,,,69,85.9421448999783,90.45663479994982,90.45663479994982,,90.45663479994982,None,None
2,0,,,,58,92.93126429995755,95.95013259997359,95.95013259997359,,95.95013259997359,None,None
3,0,,,,31,98.80565029999707,102.1219288999564,102.1219288999564,,102.1219288999564,None,None
4,0,,,,63,106.20455089997267,113.32872709998628,113.32872709998628,,113.32872709998628,None,None
5,0,,,,41,117.89311969996197,121.21110479999334,121.21110479999334,,121.21110479999334,None,None
6,0,,,,14,128.02736899995944,130.86847499996657,130.86847499996657,,130.86847499996657,None,None
7,0,,,,36,136.07599649997428,142.81687209999654,142.81687209999654,,142.81687209999654,None,None
8,0,,,,85,145.099937699968,148.4896314999787,148.4896314999787,,148.4896314999787,None,None
9,0,,,,61,153.23222589999205,157.9900798999588,157.9900798999588,,157.9900798999588,None,None


In [20]:
# Compute accuracy in attention-check questions
print('Subject:',subject_id)
question_results = []
for idx,trial in enumerate(df[['answer_index','question_response.keys']].values):
    if trial[0] == 'None' or trial[0] == '': # ignore trials without a question
        continue
    elif trial[0].lower()==trial[1].lower(): # correct trials
        question_results.append(1)
    else: # incorrect trials
        question_results.append(0)
        print(idx,trial)
question_results = np.array(question_results)
print('Accuracy:',np.mean(question_results))
print(question_results)

Subject: 44
42 ['A' 'c']
59 ['B' 'None']
75 ['C' 'a']
276 ['B' 'a']
286 ['A' 'c']
351 ['C' 'None']
411 ['D' 'a']
Accuracy: 0.825
[1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0]


### Extract sentence and question onset and offset times

In [21]:
## Extract sentence and question on/offset time data for use in GLM modelling

# Get start time for each run so that times can be adjusted to be relative to each run 'trigger' time
run_start_times = df[['fixation_text_trigger.started','key_resp.rt']].values
run_start_times = [float(pair[0])+float(pair[1]) for pair in run_start_times if pair[0] != '']
print('Run start times:',run_start_times)

# Store sentence start and end times in a dictionary with one entry for each sentence
sentence_start_times_dict = dict(zip(np.arange(1,193), np.full([num_sentences,num_runs], 0.)))
sentence_end_times_dict = dict(zip(np.arange(1,193), np.full([num_sentences,num_runs], 0.)))
question_end_times_dict = dict(zip(np.arange(1,193), np.full([num_sentences,num_runs], 0.)))
sentence_data = df[['runs.thisIndex','sentence_id','sentence.started','sentence.stopped','question_fulltext.stopped']].values
sentence_data = [sentence for sentence in sentence_data if sentence[0]!='']

# Correct presentation times to be relative to the start of each run
for trial in sentence_data:
    # extract sentence presentation data
    run_id = int(trial[0])
    sent_id = int(trial[1])
    sent_start_time = float(trial[2])
    sent_end_time = float(trial[3])
    question_end_time = float(trial[4])
    
    # compute onset and offset times
    sentence_start_times_dict[sent_id][run_id] = float(sent_start_time)-run_start_times[run_id]
    sentence_end_times_dict[sent_id][run_id] = float(sent_end_time)-run_start_times[run_id]
    question_end_times_dict[sent_id][run_id] = float(question_end_time)-run_start_times[run_id]
    
# put all end-sentence indices in a matrix, one column per run, one row per sentence
print('Sentence timing shape:',np.array(list(sentence_end_times_dict.values())).shape)

# save sentence start indices (in TR)
sentence_start_indices_dict = {}
for stim_index in sentence_start_times_dict.keys():
    indices = [int(stim_time/TR) for stim_time in sentence_start_times_dict[stim_index]]
    sentence_start_indices_dict[int(stim_index)] = indices
with open("sub-"+subject_id+"_sentence_start_indices.json", "w") as file:
    json.dump(sentence_start_indices_dict, file, indent=0)

# save sentence end indices (in TR)
sentence_end_indices_dict = {}
for stim_index in sentence_end_times_dict.keys():
    indices = [int(stim_time/TR) for stim_time in sentence_end_times_dict[stim_index]]
    sentence_end_indices_dict[int(stim_index)] = indices
with open("sub-"+subject_id+"_sentence_end_indices.json", "w") as file:
    json.dump(sentence_end_indices_dict, file, indent=0)
    
# save question end indices (in TR)
question_end_indices_dict = {}
for stim_index in question_end_times_dict.keys():
    indices = [int(stim_time/TR) for stim_time in question_end_times_dict[stim_index]]
    question_end_indices_dict[int(stim_index)] = indices
with open("sub-"+subject_id+"_question_end_indices.json", "w") as file:
    json.dump(question_end_indices_dict, file, indent=0)

Run start times: [32.693438999995124, 577.5329262000159, 1120.5391711000702, 1666.200002800033, 2213.3017072000075, 2756.5988292000256, 3301.094451699988, 3852.092805000022]
Sentence timing shape: (108, 8)


### Compute the number of characters displayed at each TR (for GLMSingle)

In [8]:
## Save question and sentence length dictionaries 

# Load dictionaries with questions and sentences
questions_file_location = file_paths_dict['neuro_root']+'2024 Fodor Dataset\\1 - Stimuli\\Fodor2024-questions-108.txt'
sents_and_ques = np.loadtxt(questions_file_location, delimiter=';', dtype='str', encoding='utf-8', skiprows=1)
questions_keys = [int(stimulus[0]) for stimulus in sents_and_ques]
questions_values = [''.join(list(stimulus[2:num_runs])) for stimulus in sents_and_ques]
sentences_values = sents_and_ques[:,num_runs]
questions_dict = dict(zip(questions_keys,questions_values))
sentences_dict = dict(zip(questions_keys,sentences_values))

# Save sentence and question length dictionary
questions_len_dict = {}
sentences_len_dict = {}
for sent_id in questions_dict.keys():
    sentences_len_dict[sent_id] = len(sentences_dict[sent_id])
    
    if questions_dict[sent_id][0:4] == 'None': # skip sents with no questions
        questions_len_dict[sent_id] = 0
    else:
        questions_len_dict[sent_id] = len(questions_dict[sent_id])
print(questions_len_dict)

# # Loop over all runs to save number of characters for each TR
sent_ids_by_time_all_runs = []
chars_displayed_by_time_all_runs = []
for run in np.arange(num_runs):

    # Save sentenence IDs at each TR
    sent_ids_by_time = np.zeros(num_TR, dtype=object)
    for sent_id in np.arange(1,109):
        sent_start_index = np.array(list(sentence_start_indices_dict.values()))[:,run][sent_id-1]
        sent_end_index = np.array(list(sentence_end_indices_dict.values()))[:,run][sent_id-1]
        question_end_index = np.array(list(question_end_indices_dict.values()))[:,run][sent_id-1]
        
        for TR_id in np.arange(sent_start_index, sent_end_index):
            sent_ids_by_time[TR_id] = sent_id
        for TR_id in np.arange(sent_end_index, question_end_index):
            sent_ids_by_time[TR_id] = 'Q'+str(sent_id)
    
    # Save character count at each TR
    chars_displayed_by_time = []
    for sent_id in sent_ids_by_time:
        if sent_id==0:
            chars_displayed_by_time.append(0)  # if no text displayed
        elif type(sent_id)==str:  # for when questions are displayed
            sent_id = int(sent_id[1:])
            chars_displayed_by_time.append(questions_len_dict[sent_id])
        else:  # for when sentences are displayed
            chars_displayed_by_time.append(sentences_len_dict[sent_id])
            
    # Append to list
    sent_ids_by_time_all_runs.append(sent_ids_by_time)
    chars_displayed_by_time_all_runs.append(chars_displayed_by_time)
    
# Save results
chars_displayed_by_time_all_runs = np.array(chars_displayed_by_time_all_runs)
np.savetxt('sub-'+subject_id+'_chars_by_TR.txt', chars_displayed_by_time_all_runs, fmt='%i')

{1: 0, 2: 0, 3: 81, 4: 117, 5: 0, 6: 0, 7: 0, 8: 99, 9: 0, 10: 90, 11: 87, 12: 94, 13: 0, 14: 0, 15: 0, 16: 148, 17: 71, 18: 79, 19: 0, 20: 94, 21: 0, 22: 104, 23: 0, 24: 0, 25: 94, 26: 119, 27: 0, 28: 92, 29: 0, 30: 92, 31: 0, 32: 114, 33: 0, 34: 0, 35: 0, 36: 0, 37: 92, 38: 0, 39: 0, 40: 0, 41: 0, 42: 71, 43: 0, 44: 0, 45: 0, 46: 97, 47: 0, 48: 0, 49: 0, 50: 100, 51: 95, 52: 86, 53: 93, 54: 0, 55: 0, 56: 0, 57: 0, 58: 74, 59: 0, 60: 0, 61: 46, 62: 0, 63: 111, 64: 0, 65: 0, 66: 72, 67: 0, 68: 0, 69: 0, 70: 67, 71: 0, 72: 67, 73: 109, 74: 0, 75: 0, 76: 0, 77: 89, 78: 0, 79: 0, 80: 120, 81: 0, 82: 0, 83: 0, 84: 0, 85: 97, 86: 0, 87: 0, 88: 0, 89: 116, 90: 0, 91: 0, 92: 0, 93: 90, 94: 80, 95: 0, 96: 81, 97: 0, 98: 119, 99: 0, 100: 0, 101: 0, 102: 0, 103: 0, 104: 0, 105: 125, 106: 0, 107: 0, 108: 116}


### Compute button press indices (for GLMsingle)

In [9]:
## Extract question answer times

# Define basic datasets needed
question_left_presses = {}
question_right_presses = {}

# Loop over runs and trials in run to extract trials with button presses
for run in np.arange(num_runs): 
    questions_data_run = df.loc[df['runs.thisIndex'] == str(run), ['question_fulltext.started','question_response.rt','question_response.keys']]
    question_left_presses_run = []
    question_right_presses_run = []
    
    for trial in questions_data_run.values: # loop over trials in each run
        if trial[2] in ['a','b']: # skip trials with no response, only save left hand presses
            response_time = float(trial[0])+float(trial[1])-run_start_times[run] # add rt to question onset time, subtract time at start of run
            response_index = int(response_time/TR)
            question_left_presses_run.append(response_index)
        elif trial[2] in ['c','d']: # skip trials with no response, only save right hand presses
            response_time = float(trial[0])+float(trial[1])-run_start_times[run] # add rt to question onset time, subtract time at start of run
            response_index = int(response_time/TR)
            question_right_presses_run.append(response_index)
            
    # Store values for each run in dict
    question_left_presses[run] = question_left_presses_run
    question_right_presses[run] = question_right_presses_run
    
# Loop over runs to store button press TR indices in array
left_press_by_time_all_runs = []
right_press_by_time_all_runs = []
for run in np.arange(num_runs):
    left_press_by_time = np.zeros(num_TR, dtype=object)
    right_press_by_time = np.zeros(num_TR, dtype=object)
    for index in np.arange(num_TR):
        # check for left presses
        if index in question_left_presses[run]:
            left_press_by_time[index] = 1
        else:
            left_press_by_time[index] = 0
        
        # check for right presses
        if index in question_right_presses[run]:
            right_press_by_time[index] = 1
        else:
            right_press_by_time[index] = 0
    
    # store results for each run
    left_press_by_time_all_runs.append(left_press_by_time)
    right_press_by_time_all_runs.append(right_press_by_time)

# Save to file
left_press_by_time_all_runs = np.array(left_press_by_time_all_runs)
right_press_by_time_all_runs = np.array(right_press_by_time_all_runs)
np.savetxt('sub-'+subject_id+'_left_press_by_TR.txt', left_press_by_time_all_runs, fmt='%i')
np.savetxt('sub-'+subject_id+'_right_press_by_TR.txt', right_press_by_time_all_runs, fmt='%i')
print('Total presses:',(left_press_by_time_all_runs+right_press_by_time_all_runs).sum())

Total presses: 40


### Archive

In [7]:
## Save numpy files with sentence onset and offset times (in seconds)

# Define storage lists 
sentence_start_indices_store = []
sentence_end_indices_store = []

# Loop over all runs and save timings to list
for run in np.arange(8):
    sentence_end_indices_np_run = sentence_end_indices_np[:,run] # extract a single run
    sentence_end_indices_np_run = sentence_end_indices_np_run[sentence_end_indices_np_run != 0.0] # remove empty time points
    sentence_end_indices_store.append(np.sort(sentence_end_indices_np_run)) # sort times when simuli were presented in that run
    
    sentence_start_indices_np_run = sentence_start_indices_np[:,run] # extract a single run
    sentence_start_indices_np_run = sentence_start_indices_np_run[sentence_start_indices_np_run != 0.0] # remove empty time points
    sentence_start_indices_store.append(np.sort(sentence_start_indices_np_run)) # sort times when simuli were presented in that run
    
# Save to file (in seconds, one column per run)
np.savetxt("sub-"+subject_id+"_sentence-start-times.txt", np.array(sentence_start_indices_store).T, delimiter=',', fmt='%s')
np.savetxt("sub-"+subject_id+"_sentence-end-times.txt", np.array(sentence_end_indices_store).T, delimiter=',', fmt='%s')

In [ ]:
## Save stimulus onset times in .1d timing files for AFNI

for sent_id in sentence_start_times_dict.keys():
    stim_times = sentence_start_times_dict[sent_id]
    filename = 'stim_times_'+str(sent_id).zfill(3)+'.1D' # pad with three spaces
    
    save_file = open(filename, "w", encoding='utf-8')
    for time in stim_times:
        if time>0: # if a stimulus was present in a given run
            save_file.write(str(time)+'\n')
        else:
            save_file.write('* *\n') # for stimuli missing in a given run
            continue
    save_file.close()